# 3. Пользовательская витрина

Одна строка — один пользователь. Витрина нужна для сегментации аудитории.

## Что считаем

Активные дни, события, прослушивания, уникальные треки, сессии, время в сессии,
Listen+, повторы, долю рекомендаций и реакции.

In [1]:
from pathlib import Path
import sys
import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SOURCE = PROJECT_ROOT / "data" / "yambda" / "flat" / "50m" / "multi_event.parquet"
MARTS = PROJECT_ROOT / "data" / "processed"
pd.set_option("display.max_columns", 30)

from src.user_mart import build_user_mart

STAGE_DB = PROJECT_ROOT / "data" / "interim" / "yambda_stage.duckdb"
MART = MARTS / "mart_user.parquet"
assert STAGE_DB.exists(), "Сначала выполните ноутбук 01_source_quality.ipynb"
con = duckdb.connect()

## Сборка витрины

Все новые столбцы этой витрины рассчитываются в `src/user_mart.py`.

In [2]:
build_user_mart(STAGE_DB, MART)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{'rows': 10000, 'sessions': 2529801}

## Портрет пользователя

In [3]:
profile = con.execute(f"""
SELECT
    count(*) AS users,
    median(events) AS median_events,
    median(listens) AS median_listens,
    median(unique_tracks) AS median_tracks,
    median(active_days) AS median_active_days,
    median(sessions) AS median_sessions,
    median(minutes_per_session) FILTER (WHERE listens > 0) AS median_minutes_per_session,
    count(*) FILTER (WHERE listens = 0) AS users_without_listens
FROM read_parquet('{MART.as_posix()}')
""").df().round(1)

profile.rename(columns={
    "users": "пользователи", "median_events": "медиана событий",
    "median_listens": "медиана прослушиваний", "median_tracks": "медиана треков",
    "median_active_days": "медиана активных дней", "median_sessions": "медиана сессий",
    "median_minutes_per_session": "медиана минут в сессии",
    "users_without_listens": "без прослушиваний",
})

,пользователи,медиана событий,медиана прослушиваний,медиана треков,медиана активных дней,медиана сессий,медиана минут в сессии,без прослушиваний
0,10000,2653.5,2571.5,964.0,86.0,156.0,33.7,762


## Сегменты по доле рекомендаций

In [4]:
segments = con.execute(f"""
SELECT
    CASE
        WHEN listens = 0 THEN 'без прослушиваний'
        WHEN recommendation_share <= 0.33 THEN 'в основном органика'
        WHEN recommendation_share <= 0.66 THEN 'смешанный'
        ELSE 'в основном рекомендации'
    END AS segment,
    count(*) AS users,
    round(avg(listen_plus_rate) * 100, 2) AS avg_listen_plus_pct,
    round(avg(minutes_per_session), 1) AS avg_minutes_per_session
FROM read_parquet('{MART.as_posix()}')
GROUP BY 1
ORDER BY users DESC
""").df()

segments.rename(columns={
    "segment": "сегмент", "users": "пользователи",
    "avg_listen_plus_pct": "средний Listen+, %",
    "avg_minutes_per_session": "среднее минут в сессии",
})

,сегмент,пользователи,"средний Listen+, %",среднее минут в сессии
0,в основном органика,3507,54.76,34.7
1,смешанный,3021,62.10,40.3
2,в основном рекомендации,2710,71.94,53.1
3,без прослушиваний,762,NaN,NaN


## Проверка

In [5]:
check = con.execute(f"""
SELECT count(*) = 10000 AS all_users,
       count(*) = count(DISTINCT uid) AS unique_key,
       sum(sessions) = 2529801 AS sessions_match,
       count(*) FILTER (WHERE listens = 0 AND listen_plus_rate IS NULL) = 762 AS nulls_valid
FROM read_parquet('{MART.as_posix()}')
""").df()
assert check.all(axis=None), "Проверка витрины не пройдена"
print("Проверка пройдена: все пользователи сохранены, ключ уникален, суммы совпадают.")

Проверка пройдена: все пользователи сохранены, ключ уникален, суммы совпадают.


## Вывод

Витрина готова для сравнения активных, органических, смешанных и рекомендательных пользователей.
Пользователей без прослушиваний следует анализировать отдельно.